# 02 — Preprocessing and Feature Selection
## AI101: Mobile Phone Price Range Classification

**Team member:** Member 2 — Preprocessing & Feature Selection

### Objective
- Create stratified 70:15:15 train/validation/test partitions.
- Handle missing numeric values and scale predictors without data leakage.
- Select the number of features using Mutual Information inside a cross-validated Logistic Regression pipeline.
- Confirm the selected configuration on validation data.
- Save split membership and configuration for Member 3.

**Target:** `price_range` (0 = low, 1 = medium, 2 = high, 3 = very high).

The test partition is reserved for final evaluation. It is not used for feature selection or hyperparameter tuning.

In [ ]:
# 1. Imports and reproducibility
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

RANDOM_STATE = 42
TARGET = "price_range"


## 2. Load data and define X and y

The supplied dataset has 20 predictor columns and one target column. The target is `price_range`. All predictors in this dataset are numeric, so a median imputer and standard scaler are appropriate baseline preprocessing steps.

In [ ]:
possible_paths = [
    Path("data/raw/Data.csv"),
    Path("../data/raw/Data.csv"),
    Path("../../data/raw/Data.csv"),
    Path("Data.csv"),
]
DATA_PATH = next((p for p in possible_paths if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Place Data.csv at data/raw/Data.csv or update DATA_PATH.")

df = pd.read_csv(DATA_PATH)
if TARGET not in df.columns:
    raise ValueError(f"Missing target column: {TARGET}")

X = df.drop(columns=TARGET)
y = df[TARGET]
print("Dataset:", df.shape)
print("Predictor count:", X.shape[1])
print("Target distribution:")
display(y.value_counts().sort_index().rename("count").to_frame())
print("Predictor data types:")
display(X.dtypes.value_counts().rename("column_count").to_frame())


## 3. Split before fitting preprocessing

We first reserve 30% of the rows, then split that portion equally into validation and test sets. Stratification preserves the class proportions in all three partitions. The original row indices are retained so the split can be reproduced and audited.

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE + 1,
    stratify=y_temp
)

split_summary = pd.DataFrame({
    "partition": ["train", "validation", "test"],
    "samples": [len(X_train), len(X_val), len(X_test)],
    "percentage": [
        len(X_train) / len(df) * 100,
        len(X_val) / len(df) * 100,
        len(X_test) / len(df) * 100,
    ],
})
display(split_summary)

print("Class proportions by partition:")
for name, target_part in [
    ("train", y_train), ("validation", y_val), ("test", y_test)
]:
    print(f"\\n{name}")
    display((target_part.value_counts(normalize=True).sort_index() * 100).rename("percentage").to_frame())

# Save original row index -> partition mapping
split_records = []
for split_name, frame in [
    ("train", X_train), ("validation", X_val), ("test", X_test)
]:
    split_records.extend(
        {"row_index": int(idx), "split": split_name} for idx in frame.index
    )
split_df = pd.DataFrame(split_records).sort_values("row_index")
split_output = Path("data/processed/split_indices.csv")
split_output.parent.mkdir(parents=True, exist_ok=True)
split_df.to_csv(split_output, index=False)
print("Saved split membership to:", split_output)


## 4. Preprocessing decisions

- **Missing values:** median imputation, because it is less sensitive to extreme values than the mean.
- **Scaling:** `StandardScaler`, because Logistic Regression is optimised numerically and features have different units/ranges.
- **Feature selection:** `SelectKBest` with Mutual Information, which can capture non-linear statistical dependence.
- **Leakage prevention:** all these steps are inside one Scikit-learn `Pipeline`. During cross-validation, each fold fits its own imputer, feature selector and scaler using only that fold's training portion.

Do not run `fit_transform` on the full dataset.

## 5. Select feature count using training-only cross-validation

We test K = 5, 8, 10, 12, 15, 18 and 20 features and Logistic Regression regularisation strengths C = 0.01, 0.1, 1, 10 and 100. The selection criterion is macro F1-score, which gives equal weight to each of the four classes.

The validation and test partitions are not passed to GridSearchCV.

In [ ]:
from functools import partial

mutual_info = partial(mutual_info_classif, random_state=RANDOM_STATE)
selection_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("select", SelectKBest(score_func=mutual_info)),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        solver="lbfgs", max_iter=3000
    )),
])

param_grid = {
    "select__k": [5, 8, 10, 12, 15, 18, 20],
    "model__C": [0.01, 0.1, 1, 10, 100],
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

search = GridSearchCV(
    selection_pipeline,
    param_grid=param_grid,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    refit=True,
    return_train_score=True,
)
search.fit(X_train, y_train)

print("Best parameters:", search.best_params_)
print("Best mean CV macro F1:", round(search.best_score_, 4))

cv_results = pd.DataFrame(search.cv_results_)
compact_results = cv_results[[
    "param_select__k", "param_model__C",
    "mean_test_score", "std_test_score", "rank_test_score"
]].sort_values(["rank_test_score", "std_test_score"])
display(compact_results.head(15))

results_path = Path("reports/results/member2_cv_results.csv")
results_path.parent.mkdir(parents=True, exist_ok=True)
compact_results.to_csv(results_path, index=False)


## 6. Confirm the selected configuration on validation data

GridSearchCV has already selected K and C using training-only cross-validation. We now evaluate that refitted best pipeline on the validation set. This is a confirmation step, not a reason to repeatedly tune against the test set.

In [ ]:
best_pipeline = search.best_estimator_
y_val_pred = best_pipeline.predict(X_val)

val_accuracy = accuracy_score(y_val, y_val_pred)
val_macro_f1 = f1_score(y_val, y_val_pred, average="macro")

print(f"Validation accuracy: {val_accuracy:.4f}")
print(f"Validation macro F1: {val_macro_f1:.4f}")
print(classification_report(y_val, y_val_pred, digits=4))


## 7. Identify the selected feature names

The selector fitted by the best cross-validated pipeline gives the selected features. These names and the best hyperparameters are saved for Member 3. The selector is learned from the training partition only.

In [ ]:
selected_mask = best_pipeline.named_steps["select"].get_support()
selected_features = X_train.columns[selected_mask].tolist()

print(f"Selected {len(selected_features)} of {X_train.shape[1]} features:")
for feature in selected_features:
    print("-", feature)

configuration = {
    "target_column": TARGET,
    "class_mapping": {
        "0": "low cost",
        "1": "medium cost",
        "2": "high cost",
        "3": "very high cost"
    },
    "random_state": RANDOM_STATE,
    "split_ratio": {"train": 0.70, "validation": 0.15, "test": 0.15},
    "cv": {"type": "StratifiedKFold", "n_splits": 5, "shuffle": True},
    "feature_selection": "SelectKBest(mutual_info_classif)",
    "selected_k": int(search.best_params_["select__k"]),
    "selected_features": selected_features,
    "logistic_regression": {
        "solver": "lbfgs",
        "multi_class": "multinomial",
        "C": float(search.best_params_["model__C"]),
        "max_iter": 3000
    },
    "best_cv_macro_f1": float(search.best_score_),
    "validation_accuracy": float(val_accuracy),
    "validation_macro_f1": float(val_macro_f1),
    "note": "Test set remains untouched. Member 3 must perform final test evaluation."
}
config_path = Path("reports/results/feature_selection_config.json")
config_path.parent.mkdir(parents=True, exist_ok=True)
config_path.write_text(json.dumps(configuration, indent=4), encoding="utf-8")
print("Saved configuration to:", config_path)


## 8. Handover to Member 3

1. Recreate the same stratified split (random states 42 and 43), or load `split_indices.csv`.
2. Use the selected feature list and Logistic Regression C value in `feature_selection_config.json`.
3. Build the final end-to-end pipeline; refit imputation, feature selection and scaling on training data only.
4. After the modelling choices are frozen, refit using training + validation data if desired.
5. Evaluate the test set once and report accuracy, per-class precision/recall/F1, macro F1 and confusion matrix.
6. Do not report validation metrics as test metrics.

**No test-set result is produced in this notebook.**